# Notebook 5: Preprocessing Pipeline
**Project:** Car Price Estimation  
**Goal:** Separate features/target, split data, encode categories, scale numerics, and build a full pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/04_engineered_data.csv')
print(f'Loaded: {df.shape}')
df.head(3)

## 1. Select Features for Modeling

In [ ]:
# Drop ID, raw CarName, redundant log cols, and the target
drop_cols = ['car_ID', 'CarName', 'model', 'log_price', 'log_enginesize',
             'log_horsepower', 'log_curbweight', 'price_segment', 'hp_category']

feature_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

X = feature_df.drop(columns=['price'])
y = df['price']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'\nFeature columns:')
print(X.columns.tolist())

## 2. Identify Numerical and Categorical Columns

In [ ]:
numerical_features   = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print(f'Numerical features  ({len(numerical_features)}): {numerical_features}')
print(f'\nCategorical features ({len(categorical_features)}): {categorical_features}')

## 3. Train / Test Split (80-20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train set : {X_train.shape[0]} samples')
print(f'Test set  : {X_test.shape[0]} samples')
print(f'Train price range: ${y_train.min():,.0f} - ${y_train.max():,.0f}')
print(f'Test  price range: ${y_test.min():,.0f} - ${y_test.max():,.0f}')

## 4. Build Preprocessing Pipeline

In [ ]:
# Numerical pipeline: impute + scale
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical pipeline: impute + one-hot encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer combining both
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline,   numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print('Preprocessing pipeline defined!')
print(preprocessor)

## 5. Fit & Transform Training Data

In [ ]:
# Fit on training, transform both train and test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'X_train processed shape: {X_train_processed.shape}')
print(f'X_test  processed shape: {X_test_processed.shape}')

# Get feature names after encoding
cat_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_feature_names)
print(f'\nTotal features after encoding: {len(all_feature_names)}')

## 6. Verify Processed Data

In [ ]:
train_df_processed = pd.DataFrame(X_train_processed, columns=all_feature_names)
print('Processed training data sample:')
print(train_df_processed.head(3))
print(f'\nAny NaNs: {train_df_processed.isnull().sum().sum()}')

## 7. Save Everything for Model Training

In [ ]:
import joblib

# Save processed arrays and targets
np.save('../data/X_train.npy', X_train_processed)
np.save('../data/X_test.npy',  X_test_processed)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_test.npy',  y_test.values)

# Save preprocessor
joblib.dump(preprocessor, '../models/preprocessor.pkl')

# Save feature names
pd.Series(all_feature_names).to_csv('../data/feature_names.csv', index=False)

# Save raw splits for reference
X_train.assign(price=y_train.values).to_csv('../data/train_split.csv', index=False)
X_test.assign(price=y_test.values).to_csv('../data/test_split.csv', index=False)

print('Saved:')
print('  data/X_train.npy, X_test.npy, y_train.npy, y_test.npy')
print('  models/preprocessor.pkl')
print('  data/feature_names.csv')
print('\nNotebook 5 Complete!')